# Fongbe ASR Training - Google Colab

Fine-tuning Whisper for Fongbe speech recognition using LoRA.

## Prerequisites
1. Runtime → GPU (T4 recommended)
2. Upload `fongbe_dataset.tar.gz` to `MyDrive/fongbe/`

## 1. Setup Environment

In [ ]:
from pathlib import Path
from google.colab import drive
import subprocess
import tarfile

# Mount Drive
drive.mount('/content/drive')

# Config paths
PROJECT_ROOT = Path('/content/drive/MyDrive/fongbe')
DATASET_TAR = PROJECT_ROOT / 'fongbe_dataset.tar.gz'
DATA_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'fongbe_asr_unified'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
REPO_URL = 'https://github.com/Appolinairee/fongbe-asr.git'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"✓ Project: {PROJECT_ROOT}")

## 2. Clone Repository

In [ ]:
import os

os.chdir('/content')
if not Path('fongbe-asr').exists():
    !git clone {REPO_URL} fongbe-asr
else:
    !cd fongbe-asr && git pull

os.chdir('fongbe-asr')
print(f"✓ Working dir: {Path.cwd()}")

## 3. Extract Dataset

In [ ]:
if not (DATA_ROOT / 'train').exists():
    if DATASET_TAR.exists():
        print(f"Extracting {DATASET_TAR.name}...")
        with tarfile.open(DATASET_TAR, 'r:gz') as tar:
            tar.extractall(PROJECT_ROOT / 'data' / 'processed')
        print("✓ Dataset extracted")
    else:
        raise FileNotFoundError(f"Dataset not found: {DATASET_TAR}")
else:
    print("✓ Dataset already extracted")

# Verify
n_train = len(list((DATA_ROOT / 'train').glob('*.arrow')))
print(f"✓ {n_train} files in train/")

## 4. Install Dependencies

In [ ]:
!pip install -q transformers accelerate peft evaluate jiwer datasets tensorboard soundfile librosa
print("✓ Dependencies installed")

## 5. Verify GPU

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Runtime → Change runtime type → GPU")

## 6. Run Training

In [ ]:
# Set paths for training script
import os
os.environ['DATASET_PATH'] = str(DATA_ROOT)
os.environ['OUTPUT_DIR'] = str(OUTPUT_ROOT / 'whisper-fongbe')

!python scripts/finetune_whisper.py

## 7. TensorBoard (Optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_ROOT / 'whisper-fongbe'}